<a href="https://colab.research.google.com/github/mariaantipina/PythonPractice/blob/main/Antipina_Numpy_Practica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

📚 Вариант 13. Фитнес-клуб (NumPy). Полный курс, 40 задач
🟢 Уровень 1. Новичок (задачи 1–10)
Что изучается: np.array, dtype, shape/size/ndim, функции создания (zeros, ones, full, empty, arange, linspace), случайные числа и воспроизводимость, память, единичные и диагональные матрицы.

Задача 1 (создание массива, dtype, shape/size/ndim). В фитнес-клубе ведётся учёт стоимости абонементов 7 клиентов: [12000, 24000, 8000, 36000, 6000, 18000, 48000] рублей. Создайте массив NumPy из этих значений. Выведите его dtype, shape, size, ndim. Дополнительно создайте второй массив из тех же значений, но с типом float32, и сравните dtype и nbytes с первым. Ожидаемый вывод: для первого массива dtype=int64, shape=(7,), size=7, ndim=1; для второго — dtype=float32, nbytes=28. В комментариях поясните: почему NumPy выбрал именно int64 для первого массива; чем shape отличается от size; почему shape у одномерного массива — это кортеж из одного элемента (7,), а не (7); что произойдёт при обращении prices[7] и почему; как nbytes связан с itemsize и size.

In [ ]:
prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000])
prices_f32 = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000], dtype=np.float32)

print(prices.dtype, prices.shape, prices.size, prices.ndim, prices.nbytes)
print(prices_f32.dtype, prices_f32.shape, prices_f32.nbytes)
print()

# int64: тип по умолчанию для целых на 64-битной платформе, вмещает все значения
# shape — длины по осям; size — общее число элементов (prod(shape))
# (7,) — кортеж из одного элемента (запятая обязательна); (7) — просто число
# prices[7] -> IndexError: индексы 0..6, выход за границу
# nbytes = itemsize * size: int64 8*7=56, float32 4*7=28
print()


**Задача 2 (подбор dtype под задачу).** В фитнес-клубе есть три вида данных: стоимость абонемента (от 1000 до 200 000 рублей); срок абонемента в месяцах (от 1 до 60); вес клиента в килограммах (от 30.0 до 250.0 с одним знаком после запятой). Для каждого вида создайте массив и подберите **минимальный** `dtype`, который вмещает все возможные значения без переполнения. Выведите `dtype`, `itemsize` и `nbytes` для каждого. Дополнительно проверьте, что произойдёт с выражением `np.array([100, 200, 250], dtype=np.uint8) + 10` — какой результат получится и почему. В комментариях объясните: почему для стоимости не стоит брать `uint8` (какой максимум он вмещает и что будет при превышении); почему для срока достаточно `uint8`, а для веса — нет; почему для веса лучше `float32`, чем `float64`, если точности в 7 знаков хватает; как связаны `dtype`, `itemsize` и `nbytes`; почему NumPy не выбрасывает исключение при переполнении и как это может привести к незаметным ошибкам в расчётах выручки клуба.

In [ ]:
cost = np.array([1000, 200000], dtype=np.uint32)
months = np.array([1, 60], dtype=np.uint8)
weight = np.array([30.0, 250.0], dtype=np.float32)

for name, arr in [("cost", cost), ("months", months), ("weight", weight)]:
    print(name, arr.dtype, arr.itemsize, arr.nbytes)

print(np.array([100, 200, 250], dtype=np.uint8) + 10)
print()

# uint8 max=255: 200000 не влезает, запись завернётся по модулю -> мусор в выручке
# срок 1..60 <= 255 -> uint8 достаточно; вес дробный -> нужен float
# float32 экономит память вдвое vs float64, 7 знаков точности для веса хватает
# dtype=тип, itemsize=байт на элемент, nbytes=itemsize*size
# NumPy не бросает исключение при переполнении (C-семантика) -> незаметные ошибки
print()



**Задача 3 (арифметические операции).** Даны стоимости абонементов четырёх клиентов `[12000, 24000, 8000, 36000]` рублей. Одной строкой без использования циклов:
- увеличьте все цены на 10% (инфляция);
- посчитайте цены со скидкой 20% (акция для новых клиентов);
- возведите цены в квадрат (условно, для расчёта бонусов);
- прибавьте к каждой цене 500 рублей (сервисный сбор).

Для каждого результата выведите массив. Ожидаемый вывод для первой операции: `[13200. 26400.  8800. 39600.]`. В комментариях объясните: почему `prices * 1.1` даёт поэлементное умножение, а не повторение, как у списков Python; что произойдёт с `dtype` после умножения на `1.1` и почему результат становится `float64`, а не остаётся `int64`; что случится, если попробовать `prices + "руб."` и почему операция выбросит ошибку, а не вернёт строковый массив.


In [ ]:
prices = np.array([12000, 24000, 8000, 36000])
print(prices * 1.1)
print(prices * 0.8)
print(prices ** 2)
print(prices + 500)
print()

# prices*1.1 — поэлементное умножение (векторизация), а не повторение списка
# умножение на float повышает dtype до float64: дробь не влезает в int64
# prices + "руб." -> TypeError/UFuncTypeError: число+строка в ufunc не определено
print()


**Задача 4 (zeros/ones/full/empty).** Фитнес-клуб выделяет буферы под учёт: массив из 30 нулей — будущие ежедневные посещения; массив из 12 единиц — веса месяцев для сезонного анализа; массив из 5 значений `-1` — маркер «абонемент не активен»; пустой массив из 3 элементов — буфер, который сразу заполните значениями `[12000, 24000, 8000]`. Для каждого выведите результат и `dtype`. Ожидаемый вывод для первого: `[0. 0. ... 0.]` с `dtype=float64`. В комментариях объясните: зачем нужен `np.empty`, если его содержимое непредсказуемо (когда это оправдано); почему `np.zeros(5)` по умолчанию даёт `float64`, а не `int64`, и как это связано с безопасностью вычислений; в каком случае `np.empty` быстрее `np.zeros` и насколько это критично для больших массивов; что произойдёт, если создать буфер через `np.empty` и случайно прочитать значения до заполнения.


In [ ]:
visits   = np.zeros(30)
weights  = np.ones(12)
inactive = np.full(5, -1)
buffer   = np.empty(3)
buffer[:] = [12000, 24000, 8000]

for name, arr in [("visits", visits), ("weights", weights),
                  ("inactive", inactive), ("buffer", buffer)]:
    print(name, arr.dtype)
print()

# empty оправдан, когда массив сразу заполняется целиком — не тратим время на зануление
# zeros по умолчанию float64 — безопасный тип для вычислений (без переполнения целых)
# empty быстрее zeros (нет зануления); для больших массивов разница заметна
# чтение empty до заполнения даёт мусор из памяти -> потенциальный баг
print()


**Задача 5 (arange, linspace).** Фитнес-клуб строит сетки для отчётов:
- `np.arange(1, 13)` — месяцы 1…12;
- `np.arange(1, 61, 6)` — сроки абонементов 1, 7, 13, …, 55 месяцев;
- `np.linspace(1000, 200000, 10)` — 10 равномерных точек стоимости от 1000 до 200 000;
- `np.linspace(0, 1, 5)` — 5 точек для нормировки.

Для каждого массива выведите результат и `len()`. Дополнительно проверьте `len(np.arange(0, 1, 0.1))` — получилось ли 10? Ожидаемый вывод для последней проверки: `10`, но может быть и `9` или `11` в зависимости от платформы из-за погрешности float. В комментариях объясните: чем `arange` отличается от `linspace`; почему `arange` с вещественным шагом может дать неожиданное количество точек; в каком случае выбирать `arange` (когда важен шаг), а в каком — `linspace` (когда важно количество точек); как избежать ошибки при построении сеток для расчёта выручки, где нужна точная длина массива.


In [ ]:
a1 = np.arange(1, 13)
a2 = np.arange(1, 61, 6)
a3 = np.linspace(1000, 200000, 10)
a4 = np.linspace(0, 1, 5)
for arr in (a1, a2, a3, a4):
    print(arr, len(arr))
print(len(np.arange(0, 1, 0.1)))
print()

# arange задаётся шагом (stop не включается); linspace — числом точек (оба конца включены)
# float-шаг в arange опасен: накопление погрешности даёт неожиданную длину (9/10/11)
# arange — когда важен шаг; linspace — когда важно количество точек
# для точной длины в расчётах выручки — linspace (или целочисленный arange)
print()


**Задача 6 (двумерные массивы, форма).** Фитнес-клуб хранит количество посещений за 3 дня по 4 типам тренировок:
```python
visits = np.array([
    [120, 150, 90, 130],   # понедельник: йога, бокс, кроссфит, пилатес
    [110, 140, 100, 125],  # вторник
    [130, 160, 95, 140],   # среда
])
```
Выведите `shape`, `ndim`, `size`, `nbytes`. Получите: посещения во вторник (`visits[1]` — одномерный массив); посещения третьего типа в среду (`visits[2, 2]` — скаляр); посещения второго типа за все дни (`visits[:, 1]` — одномерный массив). Ожидаемый вывод: `shape=(3, 4)`, `ndim=2`, `size=12`, `nbytes=96`. В комментариях объясните: почему `visits[1]` возвращает одномерный массив, а не скаляр; что произойдёт при `visits[5]` (индекс за границей); почему запись `visits[1, 2]` эффективнее, чем `visits[1][2]` (какие промежуточные объекты создаются); как `shape` связан с `ndim` и `size`.



**Задача 7 (память: itemsize и nbytes).** Фитнес-клуб хранит данные о 1 000 000 посещений. Сравните память, занимаемую: массивом `float64` (стоимость занятия); массивом `float32` (та же стоимость); массивом `uint8` (длительность в минутах, 1–120). Для каждого выведите `itemsize` и `nbytes` в мегабайтах. Ожидаемый вывод: `float64` — 7.63 МБ, `float32` — 3.81 МБ, `uint8` — 0.95 МБ. Дополнительно оцените размер Python-списка из 1 000 000 чисел через `sys.getsizeof` (только оболочки) и сравните с массивами NumPy. В комментариях объясните: почему `nbytes` не включает метаданные объекта (что ещё занимает память); почему для больших массивов выбор типа — это не микрооптимизация, а вопрос «поместится ли в RAM»; во сколько раз `uint8` экономнее `float64` и как это влияет на скорость обработки за счёт кэша процессора; в каких случаях экономия памяти важнее точности вычислений.



**Задача 8 (случайные числа, воспроизводимость).** Фитнес-клуб моделирует данные для симуляции загрузки. Создайте генератор `rng = np.random.default_rng(seed=42)` и сгенерируйте: 10 равномерных значений `rng.random(10)` — вероятности прихода клиента; 10 нормальных значений `rng.normal(1000, 300, 10)` — стоимость занятия; 10 целых значений `rng.integers(1, 61, 10)` — срок абонемента. Создайте второй генератор с тем же `seed=42` и убедитесь, что все три последовательности совпадают. Создайте третий генератор без `seed` и покажите, что значения отличаются. В комментариях объясните: зачем нужен `seed` и как он обеспечивает воспроизводимость экспериментов; чем `default_rng` отличается от старого `np.random.seed` (какой BitGenerator используется по умолчанию); почему в новом коде рекомендуется `Generator`, а не глобальный `np.random`; как `SeedSequence` помогает организовать независимые потоки случайных чисел для параллельных вычислений; в каких ситуациях отсутствие фиксированного `seed` — это хорошо, а в каких — плохо.



**Задача 9 (arange vs linspace на практике).** Фитнес-клуб строит график посещений за 30 дней. Реализуйте массив дней двумя способами: `np.arange(1, 31)` — дни 1…30; `np.linspace(1, 30, 30)` — 30 точек от 1 до 30. Проверьте `len()`, первые и последние элементы — они должны совпадать. Дополнительно найдите ситуацию, где `arange` даёт неверное количество точек из-за погрешности float: `np.arange(0, 1, 0.1)` — сколько элементов получилось? Ожидаемый ответ: 10, но может быть и другое число на некоторых платформах. В комментариях объясните: почему `linspace` надёжнее, когда важно именно количество точек; в каком случае `arange` уместнее (когда важен шаг); почему `arange` с вещественным шагом опасен в расчётах, где неверная длина массива может привести к сдвигу данных; как проверить длину массива перед использованием и что делать, если она не совпадает с ожидаемой.



**Задача 10 (единичная и диагональная матрицы).** Фитнес-клуб хранит матрицу корреляций между 4 показателями клиента (вес, возраст, число посещений, длительность тренировки). Создайте: единичную матрицу `np.eye(4)` — начальное приближение (каждый показатель коррелирует сам с собой); диагональную матрицу `np.diag([1.0, 0.8, 0.6, 0.9])` — дисперсии показателей; матрицу со смещённой диагональю `np.diag([0.5, 0.3], k=1)` — корреляции соседних показателей. Выведите все три. Ожидаемый вывод для первой: диагональ из единиц, остальные нули. В комментариях объясните: чем `np.eye(4)` отличается от `np.identity(4)` (какой параметр позволяет задать вторую размерность); что делает параметр `k` в `np.diag` (положительное значение — диагональ выше главной, отрицательное — ниже); в каком случае диагональная матрица лучше обычной (память и скорость умножения, если большинство элементов нули); почему `np.eye` по умолчанию даёт `float`, а не `int`, и когда это важно.

---


## 🔵 Уровень 2. Ученик (задачи 11–18)

**Что изучается:** индексация, срезы, логическая и fancy-индексация, view vs copy, `reshape`, `ravel`/`flatten`, транспонирование и перестановка осей.

**Задача 11 (индексация).** Дан массив цен `prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000])`. Получите: цену первого абонемента (`prices[0]`); цену последнего (`prices[-1]`); цену третьего (`prices[2]`). Дана матрица `visits` формы `(3, 4)` из задачи 6. Получите: посещения второго типа в понедельник (`visits[0, 1]`); все посещения во вторник (`visits[1]`); все посещения второго типа (`visits[:, 1]`). Ожидаемые результаты: `prices[0] = 12000`, `prices[-1] = 48000`, `visits[0, 1] = 150`, `visits[1] = [110, 140, 100, 125]`, `visits[:, 1] = [150, 140, 160]`. В комментариях объясните: почему `visits[1]` возвращает одномерный массив, а не скаляр; что произойдёт при `visits[5]` (какое исключение и с каким сообщением); почему отрицательные индексы работают и для многомерных массивов; что вернёт `visits[-1, -1]` и почему это последний элемент последней строки.


In [ ]:
prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000])
visits = np.array([
    [120, 150, 90, 130],
    [110, 140, 100, 125],
    [130, 160, 95, 140],
])
print(prices[0], prices[-1], prices[2])
print(visits[0, 1])
print(visits[1])
print(visits[:, 1])
print(visits[-1, -1])
print()

# visits[1] — строка целиком -> одномерный массив, а не скаляр
# visits[5] -> IndexError: index 5 is out of bounds for axis 0 with size 3
# отрицательные индексы работают и в многомерном случае (отсчёт с конца)
# visits[-1,-1] — последний элемент последней строки = 140
print()



**Задача 12 (срезы).** Дан массив цен на 30 абонементов: `prices = np.arange(30) * 1000 + 5000`. Получите: первые 7 (`prices[:7]`); последние 7 (`prices[-7:]`); с 10-го по 20-й (`prices[10:20]`); каждый второй (`prices[::2]`); массив в обратном порядке (`prices[::-1]`). Дана матрица `visits` формы `(3, 4)`. Получите: первые две строки (`visits[:2, :]`); второй и третий столбцы (`visits[:, 1:3]`); подматрицу `2×2` из левого верхнего угла (`visits[:2, :2]`). В комментариях объясните: включается ли правая граница среза (что вернёт `prices[10:20]` — 10 или 11 элементов); что делает `prices[::-1]` (как получается обратный порядок); почему срез возвращает view, а не копию; как проверить, что `prices[:7].base is prices`, и что это означает; в каком случае срез с шагом `[::2]` может быть опасен при изменении данных.


In [ ]:
prices = np.arange(30) * 1000 + 5000
visits = np.array([
    [120, 150, 90, 130],
    [110, 140, 100, 125],
    [130, 160, 95, 140],
])
print(prices[:7])
print(prices[-7:])
print(prices[10:20], len(prices[10:20]))
print(prices[::2])
print(prices[::-1])
print(visits[:2, :])
print(visits[:, 1:3])
print(visits[:2, :2])
print(prices[:7].base is prices)
print()

# правая граница не включается: prices[10:20] — 10 элементов (10..19)
# prices[::-1] — обратный порядок за счёт отрицательного шага
# срез возвращает view: prices[:7].base is prices -> True (общий буфер)
# шаговый срез [::2] тоже view -> изменение через него меняет оригинал (баг-риск)
print()


**Задача 13 (логическая индексация).** Дан массив цен `prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000, 4000, 60000])`. Получите: премиум-абонементы (цена выше 20000) (`prices[prices > 20000]`); дешёвые (менее 10000) (`prices[prices < 10000]`); средние от 10000 до 20000 включительно (`prices[(prices >= 10000) & (prices <= 20000)]`); очень дешёвые или очень дорогие (`prices[(prices < 5000) | (prices > 40000)]`). Дополнительно замените все цены ниже 5000 на 5000 (`prices[prices < 5000] = 5000`) и выведите массив. В комментариях объясните: почему при комбинировании условий нужны скобки вокруг каждого условия; почему нельзя использовать `and`/`or` вместо `&`/`|` (какая ошибка возникнет); почему логическая индексация возвращает копию, а не view; как `np.where` мог бы заменить последнюю операцию и в каком случае он удобнее.


In [ ]:
prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000, 4000, 60000])
print(prices[prices > 20000])
print(prices[prices < 10000])
print(prices[(prices >= 10000) & (prices <= 20000)])
print(prices[(prices < 5000) | (prices > 40000)])
prices[prices < 5000] = 5000
print(prices)
# np.where(prices < 5000, 5000, prices) — эквивалент без изменения на месте
print()

# скобки обязательны: & и | приоритетнее сравнений
# and/or не работают поэлементно -> ValueError: truth value ... is ambiguous
# логическая индексация возвращает копию, а не view
# np.where удобнее, когда нужен новый массив и нельзя менять оригинал
print()


**Задача 14 (fancy indexing).** Дан массив цен `prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000])`. Получите: цены абонементов с индексами 0, 2, 5 (`prices[[0, 2, 5]]`); те же абонементы в обратном порядке (`prices[[5, 2, 0]]`); абонементы 0, 0, 1, 1, 2 с повторениями (`prices[[0, 0, 1, 1, 2]]`). Дана матрица `visits` формы `(3, 4)`. Получите: строки 0 и 2 (`visits[[0, 2]]`); столбцы 1 и 3 (`visits[:, [1, 3]]`); диагональные элементы `visits[0, 0], visits[1, 1], visits[2, 2]` (`visits[[0, 1, 2], [0, 1, 2]]`); прямоугольную подматрицу через `np.ix_([0, 2], [1, 3])` — сравните с `visits[[0, 2], [1, 3]]` и объясните разницу. В комментариях объясните: чем fancy indexing отличается от среза; почему fancy indexing всегда возвращает копию; зачем нужен `np.ix_` (когда без него выбираются только диагональные элементы); что произойдёт, если индексы выйдут за границу массива.


In [ ]:
prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000])
visits = np.array([
    [120, 150, 90, 130],
    [110, 140, 100, 125],
    [130, 160, 95, 140],
])
print(prices[[0, 2, 5]])
print(prices[[5, 2, 0]])
print(prices[[0, 0, 1, 1, 2]])
print(visits[[0, 2]])
print(visits[:, [1, 3]])
print(visits[[0, 1, 2], [0, 1, 2]])
print(visits[np.ix_([0, 2], [1, 3])])
print(visits[[0, 2], [1, 3]])
print()

# fancy indexing — выборка списком индексов; всегда возвращает копию
# visits[[0,2],[1,3]] берёт пары (0,1) и (2,3), а не подматрицу
# np.ix_ строит открытую сетку -> прямоугольная подматрица 2x2
# индексы вне границ -> IndexError
print()

**Задача 15 (view vs copy).** Дан массив цен `prices = np.arange(10) * 1000 + 5000`. Создайте: `view = prices[2:5]` — срез; `copy = prices[2:5].copy()` — копия. Измените `view[0] = 999999` и `copy[0] = 111111`. Выведите `prices`, `view`, `copy`. Проверьте: `view.base is prices` → `True`; `copy.base is None` → `True`; `np.shares_memory(prices, view)` → `True`; `np.shares_memory(prices, copy)` → `False`. В комментариях объясните: почему NumPy возвращает view для срезов, а не копию (какая экономия достигается); в каком случае это приводит к неожиданным ошибкам (например, при передаче среза в функцию, которая меняет данные); как защититься — вызвать `.copy()`; почему view экономит память, но опасен для больших массивов, если держит ссылку на оригинал (утечка памяти); как проверить, является ли массив view или копией.


In [ ]:
prices = np.arange(10) * 1000 + 5000
view = prices[2:5]
copy = prices[2:5].copy()
view[0] = 999999
copy[0] = 111111
print(prices)
print(view)
print(copy)
print(view.base is prices)
print(copy.base is None)
print(np.shares_memory(prices, view))
print(np.shares_memory(prices, copy))
print()

# срез — view: экономит память, буфер не копируется
# опасно: функция, меняющая срез, испортит оригинал
# .copy() — независимая копия, оригинал не затрагивается
# view держит ссылку на оригинальный буфер -> огромный массив не освободится
# проверка: .base is None (копия), np.shares_memory (общий буфер)
print()


**Задача 16 (reshape).** Дан массив из 12 цен: `prices = np.arange(12) * 1000 + 5000`. Переформируйте его: в матрицу `3×4` (`prices.reshape(3, 4)`); в матрицу `2×6` (`prices.reshape(2, 6)`); в трёхмерный массив `2×2×3` (`prices.reshape(2, 2, 3)`); в матрицу `3×4` с автоматическим определением второй размерности (`prices.reshape(3, -1)`). Попробуйте `prices.reshape(5, 5)` — получите ошибку и объясните, почему. В комментариях объясните: почему `reshape` не меняет данные, а только форму (что происходит с блоком памяти); когда `reshape` возвращает view, а когда копию (проверьте `.base` после транспонирования); зачем нужен `-1` и что произойдёт, если указать две `-1`; как `reshape` связан с порядком хранения в памяти (`order='C'` vs `order='F'` — в каком порядке читаются элементы).



**Задача 17 (ravel и flatten).** Дана матрица `visits` формы `(3, 4)`. Получите одномерный массив тремя способами: `visits.ravel()`; `visits.flatten()`; `visits.reshape(-1)`. Сравните `.base` у каждого результата: у `ravel` и `reshape(-1)` — ссылка на `visits`, у `flatten` — `None`. Измените элемент в `ravel` и убедитесь, что `visits` изменилась; измените элемент в `flatten` и убедитесь, что `visits` не изменилась. В комментариях объясните: в каком случае выбирать `ravel` (когда не нужно изменять), а в каком `flatten` (когда нужна независимая копия); почему `flatten` медленнее (какие данные копируются); что такое «непрерывный массив в C-порядке» и почему `ravel` иногда тоже возвращает копию (например, после транспонирования); как это связано с производительностью при обработке больших массивов.



**Задача 18 (транспонирование и перестановка осей).** Дан трёхмерный массив `data` формы `(2, 3, 4)` — например, 2 филиала клуба × 3 дня × 4 типа тренировок. Получите: `data.T` — обратный порядок осей, форма `(4, 3, 2)`; `np.transpose(data, (1, 0, 2))` — поменять оси 0 и 1, форма `(3, 2, 4)`; `np.swapaxes(data, 0, 2)` — поменять оси 0 и 2, форма `(4, 3, 2)`; `np.moveaxis(data, 0, -1)` — переместить ось 0 в конец, форма `(3, 4, 2)`. Для двумерной `visits` формы `(3, 4)` сравните `visits.T` и `visits.transpose()` — результат должен совпадать. В комментариях объясните: почему транспонирование возвращает view, а не копию (что меняется в метаданных); как `moveaxis` отличается от `swapaxes` (когда нужен каждый); в каком случае транспонирование реально копирует данные (например, после `visits[::2]`, когда массив не непрерывный); как проверить, является ли результат view или копией.

---



## 🟣 Уровень 3. Мастер (задачи 19–24)

**Что изучается:** broadcasting (правила, ошибки, практика), `axis`, `keepdims`, универсальные функции (`ufunc`), методы `reduce`/`accumulate`/`outer`, условная логика (`where`/`select`/`clip`).

**Задача 19 (правила broadcasting).** Дана матрица `visits` формы `(3, 4)` (3 дня × 4 типа тренировок) и вектор поправок `offset` формы `(4,)` (например, изменение числа посещений по типам). Сложите их: `visits + offset`. Объясните, как NumPy растянул `offset` на все три строки (какие размерности были добавлены виртуально). Затем создайте: `offset_col` формы `(3, 1)` — сложите с `visits` (объясните, как теперь растягивается `offset_col`); `scalar` — сложите с `visits` (объясните, почему скаляр всегда совместим); `wrong` формы `(3,)` — попробуйте сложить с `visits` и получите ошибку (объясните, почему `(3,)` несовместима с `(3, 4)`, а `(4,)` — совместима). Для каждой пары выведите формы и результат. В комментариях объясните три правила broadcasting: выравнивание размерностей (добавление единиц слева); совместимость по осям (равны или одна равна 1); результирующая форма — поэлементный максимум; как прочитать сообщение об ошибке `operands could not be broadcast together with shapes (3,4) (3,)` и что оно означает; в каком случае broadcasting может неявно создать огромный массив и как это предотвратить.


In [ ]:
visits = np.array([
    [120, 150, 90, 130],
    [110, 140, 100, 125],
    [130, 160, 95, 140],
])
offset = np.array([10, 20, 30, 40])
print((visits + offset))
offset_col = np.array([[10], [20], [30]])
print(visits + offset_col)
print(visits + 5)
wrong = np.array([1, 2, 3])
try:
    visits + wrong
except ValueError as e:
    print(e)
print()

# (4,) растягивается на все 3 строки: добавлена ось слева -> (1,4) -> (3,4)
# offset_col (3,1) растягивается по столбцам -> (3,4)
# скаляр всегда совместим с любой формой
# (3,) vs (3,4): справа 3 != 4 -> operands could not be broadcast together
# правила: выравнивание справа, размеры равны или один=1, результат — максимум по осям
# неявный огромный массив: (1000000,1)*(1,1000000) -> 10^12 элементов, следи за формами
print()


**Задача 20 (broadcasting на практике: нормализация).** Дана матрица характеристик клиентов формы `(365, 5)` — 365 дней × 5 характеристик (вес, возраст, число посещений, длительность тренировки, потраченные калории). Вычислите: среднее по каждой характеристике (`data.mean(axis=0)` — форма `(5,)`); стандартное отклонение (`data.std(axis=0)` — форма `(5,)`); центрирование: `centered = data - data.mean(axis=0)`; стандартизацию: `standardized = centered / data.std(axis=0)`. Проверьте, что после стандартизации среднее по каждому столбцу близко к 0, а стандартное отклонение — к 1. В комментариях объясните: почему `data - data.mean(axis=0)` работает без явного `reshape` (как broadcasting растягивает `(5,)` до `(365, 5)`); что произойдёт, если перепутать `axis=0` и `axis=1` (какая ошибка или неверный результат); как broadcasting экономит память (не создаёт копию среднего на 365 строк); как то же самое сделать через `keepdims=True` и в каком случае это удобнее; зачем вообще нужна стандартизация перед анализом клиентов.


In [ ]:
rng = np.random.default_rng(0)
data = rng.normal(100, 20, (365, 5))
mu = data.mean(axis=0)
sd = data.std(axis=0)
centered = data - mu
standardized = centered / sd
print(data[:6])
print(np.allclose(standardized.mean(axis=0), 0, atol=1e-12))
print(np.allclose(standardized.std(axis=0), 1, atol=1e-12))
print()

# (5,) растягивается до (365,5) без reshape — broadcasting
# путаница axis=0/1 даст неверную нормализацию (по дням вместо признаков)
# копия среднего не создаётся -> экономия памяти
# keepdims=True удобнее: data - data.mean(0, keepdims=True)
# стандартизация нужна для сравнения признаков с разными масштабами
print()


**Задача 21 (ось агрегации `axis`).** Дана матрица `visits` формы `(3, 4)`. Вычислите: `visits.sum()` — одно число; `visits.sum(axis=0)` — форма `(4,)`, сумма по дням (по столбцам); `visits.sum(axis=1)` — форма `(3,)`, сумма по типам тренировок (по строкам). Повторите для `min`, `max`, `mean`, `std`. В комментариях объясните: почему `axis=0` «сворачивает» строки (дни), а `axis=1` — столбцы (типы тренировок); как запомнить правило («`axis` — это то, что исчезает»); что произойдёт при `axis=2` для двумерного массива (какая ошибка); как сформулировать задачу «среднее число посещений по типам тренировок за все дни» через `axis`; как работает агрегация по кортежу осей (`axis=(0, 1)`) и в каком случае это нужно.


In [ ]:
visits = np.array([
    [120, 150, 90, 130],
    [110, 140, 100, 125],
    [130, 160, 95, 140],
])
print(visits.sum())
print(visits.sum(axis=0))
print(visits.sum(axis=1))
print(visits.min(axis=0), visits.max(axis=1), visits.mean(axis=0), visits.std(axis=1))
print(visits.sum(axis=(0, 1)))
print()

# axis=0 сворачивает строки (сумма по дням для каждого типа)
# axis=1 сворачивает столбцы (сумма по типам внутри дня)
# правило "axis — это то, что исчезает"
# axis=2 для 2D -> AxisError: axis 2 is out of bounds
# "среднее по типам за все дни" -> visits.mean(axis=0)
# axis=(0,1) — агрегация сразу по двум осям -> скаляр
print()


**Задача 22 (`keepdims` и broadcasting).** Дана `visits` формы `(3, 4)`. Вычислите среднее по строкам (по типам тренировок внутри дня): без `keepdims`: `row_means = visits.mean(axis=1)` — форма `(3,)`; с `keepdims=True`: `row_means_keep = visits.mean(axis=1, keepdims=True)` — форма `(3, 1)`. Попробуйте вычесть `row_means` из `visits` — получите ошибку broadcasting (объясните, почему). Вычтите `row_means_keep` — работает. В комментариях объясните: зачем нужен `keepdims` (какая размерность сохраняется); почему `(3,)` и `(3, 1)` — разные формы с точки зрения broadcasting; в каком случае `keepdims` экономит одну строку кода (когда сразу нужно использовать результат в операциях); как то же самое сделать через `[:, np.newaxis]`; почему `keepdims` не меняет данные, а только форму результата; что произойдёт с формой, если применить `keepdims=True` дважды.



**Задача 23 (универсальные функции ufunc).** Дан массив цен `prices = np.array([12000, 24000, 8000, 36000, 6000])`. Примените: `np.add(prices, 500)` — сервисный сбор; `np.multiply(prices, 1.1)` — налог 10%; `np.power(prices, 2)` — квадрат; `np.sqrt(prices)` — корень; `np.exp(prices / 100000)` — экспонента; `np.log(prices)` — логарифм. Используйте параметры ufunc: `out` — запишите результат в заранее созданный массив; `where` — примените операцию только там, где `prices > 15000`. В комментариях объясните: что такое ufunc (чем отличается от обычной функции Python); почему `np.add(a, b)` и `a + b` эквивалентны; зачем нужны `out` (экономия памяти) и `where` (условное применение); как ufunc связаны с broadcasting; в каком случае `np.log` вернёт `-inf` или `nan` (при нуле и отрицательных значениях) и как это обработать; какие ещё ufunc вы знаете (`np.sin`, `np.abs`, `np.maximum`, `np.mod`) и где они применяются в расчётах клуба.


**Задача 24 (условная логика: `where`, `select`, `clip`).** Дан массив цен `prices = np.array([4000, 8000, 12000, 24000, 36000, 48000, 60000])`. Реализуйте: `np.where(prices < 5000, 5000, prices)` — поднимите минимальную цену до 5000; `np.where(prices > 50000, 50000, prices)` — обрежьте максимальную цену. Для категоризации используйте `np.select`: условия — цена < 10000, от 10000 до 30000, больше 30000; значения — «базовый», «стандарт», «премиум»; значение по умолчанию — «?». Наконец, `np.clip(prices, 5000, 50000)` — обрежьте с обеих сторон. Дополнительно реализуйте `np.add.reduce` и `np.add.accumulate` для массива ежедневных посещений `visits_per_day = np.array([12, 15, 11, 18, 14, 13, 16])`. В комментариях объясните: чем `where` отличается от `select` (когда два варианта, когда много); почему при перекрывающихся условиях `select` берёт первое (как это использовать для приоритетов); в каком случае `clip` лучше `where` (краткость, скорость, читаемость); чем `reduce` отличается от `accumulate` (свёртка в одно значение vs промежуточные результаты); что такое `identity` у ufunc (`np.add.identity == 0`) и зачем он нужен.

---



## 🟠 Уровень 4. Архитектор (задачи 25–30)

**Что изучается:** сортировка (`sort`, `argsort`), частичная сортировка (`partition`, `argpartition`), бинарный поиск (`searchsorted`), многоуровневая сортировка (`lexsort`), `unique` и операции над множествами, `isin`.

**Задача 25 (сортировка).** Дан массив цен `prices = np.array([12000, 24000, 8000, 36000, 6000, 18000, 48000])`. Получите: отсортированный по возрастанию (`np.sort(prices)`); отсортированный по убыванию (`np.sort(-prices) * -1` или `np.sort(prices)[::-1]`); отсортируйте на месте через `prices.sort()` и убедитесь, что исходный массив изменился. Дана матрица `visits` формы `(3, 4)`. Отсортируйте: по умолчанию (`axis=-1`, по строкам); по оси 0 (`axis=0`, по столбцам); с `axis=None` (уплощённый массив). В комментариях объясните: чем `np.sort(arr)` отличается от `arr.sort()` (какой возвращает копию, какой меняет на месте); почему сортировка по умолчанию выполняется по последней оси; как NumPy обрабатывает NaN при сортировке (проверьте, добавив `np.nan` — куда попадут NaN); какой алгоритм сортировки используется по умолчанию и как выбрать стабильный (`kind='stable'`) — в каком случае это важно (например, при сортировке по нескольким ключам).




**Задача 26 (`argsort` и топ-N).** Дан массив посещений на 10 клиентов: `visits = np.array([12, 15, 9, 13, 11, 14, 10, 8, 16, 9])`. Получите: индексы, которые отсортировали бы массив (`np.argsort(visits)`); примените индексы к массиву (`visits[np.argsort(visits)]`) и убедитесь, что результат совпадает с `np.sort(visits)`. Дополнительно даны имена клиентов `names = np.array(["A","B","C","D","E","F","G","H","I","J"])`. Получите клиентов, отсортированных по числу посещений (`names[np.argsort(visits)]`). Найдите топ-3 самых активных: `names[np.argsort(-visits)[:3]]`. В комментариях объясните: зачем нужен `argsort`, если есть `sort` (как он позволяет сортировать параллельные массивы); как с его помощью отсортировать имена по числу посещений, не теряя соответствия; почему `argsort(-visits)` даёт порядок по убыванию; что такое «непрямая сортировка» и в каких задачах она необходима (ранжирование, топ-N, сортировка по нескольким полям).



**Задача 27 (`partition` и `argpartition`).** Дан массив из 1 000 000 случайных значений посещений. Найдите 10 самых активных клиентов: через полную сортировку (`np.sort(visits)[-10:]`); через `np.partition(visits, -10)[-10:]`; через `np.argpartition(visits, -10)[-10:]` — индексы. Замерьте время каждого подхода через `time.perf_counter()`. В комментариях объясните: почему `partition` работает за O(n), а `sort` — за O(n log n); что гарантирует `partition` (k-й элемент на своём месте, слева — меньше, справа — больше, но порядок внутри частей не определён); во сколько раз `partition` быстрее `sort` для 1 000 000 элементов (по экспериментальным данным); почему `argpartition` + `argsort` дают корректный топ-N (что нужно сделать после `argpartition`, чтобы упорядочить топ); в каком случае разница в скорости критична (интерактивные приложения, реальное время).



**Задача 28 (`searchsorted` и категоризация).** Даны границы ценовых категорий: `bounds = np.array([0, 10000, 20000, 40000, 100000])` (базовый, стандарт, комфорт, премиум, VIP). Даны цены: `prices = np.array([6000, 12000, 18000, 24000, 36000, 48000])`. Определите категорию каждой цены: `np.searchsorted(bounds, prices)` — индексы; `np.searchsorted(bounds, prices, side='right')` — сравните. Затем подсчитайте частоты: `np.bincount(np.searchsorted(bounds, prices))`. В комментариях объясните: что возвращает `searchsorted` (позицию для вставки, а не индекс найденного элемента); чем `side='left'` отличается от `side='right'` на граничных значениях (например, для цены ровно 20000); как `searchsorted` + `bincount` заменяют `histogram` (в каком случае это удобнее); как `searchsorted` работает с большими массивами за O(log n) (бинарный поиск) и почему это важно для категоризации миллионов абонементов.



**Задача 29 (`lexsort` и многоуровневая сортировка).** Даны три массива: `types` (типы абонементов), `prices` (цены), `months` (сроки). Отсортируйте данные: сначала по типу абонемента, затем по цене (`np.lexsort((prices, types))`); сначала по цене, затем по сроку (`np.lexsort((months, prices))`). Примените индексы ко всем трём массивам и выведите результат. В комментариях объясните: почему последний ключ в `lexsort` — основной (порядок аргументов — от второстепенного к главному); чем `lexsort` отличается от `np.argsort(structured, order=...)` (какой подход удобнее для отдельных массивов, какой для структурированных); в каком случае удобнее структурированный массив (когда данные связаны и их много); как `lexsort` связан со стабильной сортировкой (какой алгоритм используется); какие ещё задачи решает многоуровневая сортировка (ранжирование клиентов по нескольким критериям, группировка по типам абонементов).



**Задача 30 (`unique` и операции над множествами).** Даны два массива типов тренировок из двух залов:
```python
hall1 = np.array(["йога", "бокс", "кроссфит", "йога", "пилатес"])
hall2 = np.array(["бокс", "кроссфит", "стретчинг", "йога"])
```
Получите: уникальные типы каждого зала (`np.unique`); уникальные с количеством вхождений (`np.unique(..., return_counts=True)`); пересечение (`np.intersect1d(hall1, hall2)`); объединение (`np.union1d`); разность (`np.setdiff1d(hall1, hall2)`); симметричную разность (`np.setxor1d`); проверку принадлежности (`np.isin(hall1, hall2)`). В комментариях объясните: почему все операции возвращают отсортированные уникальные значения (как это связано с внутренней реализацией через сортировку); чем `assume_unique=True` ускоряет работу (какой этап пропускается); чем `np.isin` отличается от `np.in1d` (какой рекомендуется и почему — форма результата, поддержка многомерных массивов); как `np.unique(..., axis=0)` работает для двумерных массивов (что считается уникальным — строки или элементы); в каких задачах клуба применяются операции над множествами (сравнение расписаний, поиск пересечений тренировок, анализ дублей).

---



## 🔴 Уровень 5. Эксперт (задачи 31–35)

**Что изучается:** обработка NaN, статистика (среднее, медиана, квантили, `ddof`), корреляции и ковариации, линейная алгебра, генераторы случайных чисел и распределения.

**Задача 31 (обработка NaN).** Дан массив весов клиентов с пропусками: `weights = np.array([75.5, 82.3, np.nan, 68.0, np.nan, 90.1])`. Проверьте: `np.isnan(weights)` — маска; `np.sum(np.isnan(weights))` — количество NaN; `weights.sum()` — результат `nan`; `np.nansum(weights)`, `np.nanmean(weights)`, `np.nanstd(weights)` — игнорируют NaN. Дополнительно: `np.nan_to_num(weights, nan=0.0)` — замените NaN; `np.isfinite(weights)` — проверка на конечность. В комментариях объясните: почему обычные агрегаты «заражаются» NaN (как NaN распространяется при сложении); когда лучше `np.nan_to_num` (когда нужно сохранить длину массива), а когда `np.nanmean` (когда нужна статистика без пропусков); что вернёт `np.nanmean` для массива из одних NaN (и как это отследить); почему `np.isnan` не работает с целочисленными массивами (какая ошибка возникнет и почему); как обработать NaN в двумерном массиве через `np.nanmean(axis=0)` и что произойдёт с полностью пустым столбцом.



**Задача 32 (статистика).** Дан массив цен за год формы `(365,)`. Вычислите: `mean`, `median`, `std`, `var`, `min`, `max`, `ptp`. Сравните `std(ddof=0)` и `std(ddof=1)` — объясните разницу. Вычислите `np.percentile(prices, [25, 50, 75])` — квартили; `np.quantile(prices, [0.1, 0.9])` — децили. Дополнительно для матрицы характеристик формы `(365, 5)` вычислите среднее и стандартное отклонение по каждой характеристике (`axis=0`) и по каждому дню (`axis=1`). В комментариях объясните: когда `ddof=1` (выборочное) предпочтительнее `ddof=0` (генеральное) — в каком случае какая формула корректна; почему медиана устойчива к выбросам, а среднее — нет (пример: один VIP-абонемент за 500 000 сильно смещает среднее, но не медиану); как `quantile` и `percentile` связаны (какой параметр отличается); почему `np.ptp` — это `max - min` и в каком случае он полезен (оценка разброса цен); как интерпретировать квартили для сегментации клиентов.



**Задача 33 (корреляции и ковариации).** Дана матрица характеристик формы `(365, 5)` — 365 дней × 5 характеристик. Вычислите: центрирование `centered = data - data.mean(axis=0)`; ковариационную матрицу `cov = centered.T @ centered / (365 - 1)`; стандартизацию `standardized = centered / data.std(axis=0)`; корреляционную матрицу `corr = standardized.T @ standardized / (365 - 1)`. Проверьте, что `corr` имеет единицы на диагонали и симметрична. Дополнительно вычислите корреляцию двух конкретных характеристик вручную через `np.corrcoef`. В комментариях объясните: чем ковариация отличается от корреляции (единицы измерения, диапазон); почему `centered.T @ centered` — это матричное умножение, а не поэлементное (что вычисляется); как broadcasting используется в центрировании; зачем нужна стандартизация перед корреляцией (чтобы убрать масштаб); как интерпретировать корреляционную матрицу для анализа клиентов (какие характеристики связаны).



**Задача 34 (линейная алгебра).** Дана система линейных уравнений `A @ x = b`:
```python
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([5.0, 10.0])
```
Решите систему: через `np.linalg.solve(A, b)`; через `np.linalg.inv(A) @ b` — сравните результаты и объясните, какой способ лучше. Дополнительно вычислите: `np.linalg.det(A)` — определитель; `np.linalg.eig(A)` — собственные значения и векторы; `np.linalg.norm(b)` — евклидова норма; `np.linalg.norm(b, ord=1)` и `np.linalg.norm(b, ord=np.inf)` — другие нормы. В комментариях объясните: почему `solve` предпочтительнее `inv(A) @ b` (скорость, устойчивость, накопление ошибок при обращении матрицы); что такое собственные значения (на примере: для диагональной матрицы — сами диагональные элементы); в каком случае матрица не имеет обратной (определитель равен нулю — что это означает геометрически); как нормы используются для оценки сходства клиентов (косинусное сходство характеристик, норма разности векторов).



**Задача 35 (случайные числа и распределения).** Создайте `rng = np.random.default_rng(seed=42)`. Сгенерируйте: 1000 нормальных цен (`rng.normal(15000, 5000, 1000)`); 1000 равномерных возрастов (`rng.uniform(16, 80, 1000)`); 1000 биномиальных «посещений» (`rng.binomial(10, 0.6, 1000)`); 1000 пуассоновских «клиентов в день» (`rng.poisson(50, 1000)`). Для каждого распределения выведите среднее и стандартное отклонение и сравните с теоретическими значениями. Дополнительно: `rng.choice(100, size=10, replace=False)` — выборка клиентов без повторений; `rng.shuffle(members)` — перемешивание клиентов; `rng.permutation(10)` — новая перестановка. В комментариях объясните: чем `choice(replace=False)` отличается от `permutation` (когда нужна выборка, когда перестановка); зачем нужен `SeedSequence` для параллельных вычислений (как избежать корреляции между потоками); почему `default_rng` лучше старого `np.random.seed` (какой BitGenerator, какие свойства); в каком случае `rng.normal` может дать отрицательные цены и как это исправить (через `clip` или `lognormal` — какое распределение естественнее для цен).

---



## ⚫ Уровень 6. Гуру (задачи 36–40)

**Что изучается:** векторизация и производительность, операции на месте, FFT и обработка сигналов, полиномы и интерполяция, `einsum` и структурированные массивы.

**Задача 36 (векторизация vs цикл).** Даны два массива из 10 000 000 цен занятий. Реализуйте вычисление `a + b` тремя способами: чистый Python-цикл `[a[i] + b[i] for i in range(n)]`; NumPy: `a + b`; NumPy с `out`: `np.add(a, b, out=result)`. Замерьте время через `time.perf_counter()` (предварительно прогрейте, чтобы избежать влияния первого запуска). В комментариях объясните: почему NumPy в десятки раз быстрее (что происходит на уровне C, какие оптимизации); как непрерывная память и SIMD-инструкции влияют на скорость; в каком случае `out` даёт выигрыш (когда массив большой и создание нового дорого); почему `a + b` создаёт новый массив, а `np.add(a, b, out=a)` — нет (и когда это опасно — если `a` нужен дальше); как векторизация связана с broadcasting (векторизация — частный случай); какие ещё операции клуба можно векторизовать вместо циклов.



**Задача 37 (операции на месте).** Дана матрица цен `prices` формы `(1000, 1000)`. Сравните: `prices = prices * 1.1` — создаёт новый массив; `prices *= 1.1` — на месте, без нового массива. Замерьте время и потребление памяти (через `nbytes` до и после). Дополнительно: `np.multiply(prices, 1.1, out=prices)` — ufunc с `out`; `np.clip(prices, 1000, 200000, out=prices)` — обрезка на месте. В комментариях объясните: почему `prices *= 1.1` быстрее и экономнее (какая память не выделяется); почему `prices[:] = prices * 1.1` **не** экономит память (всё равно создаётся временный массив для правой части); в каком случае операции на месте вредны (когда нужен исходный массив для отладки или сравнения); как `out` в ufunc связан с операциями на месте (это один и тот же механизм); какие ещё операции клуба можно выполнять на месте (нормализация, обрезка, масштабирование).



**Задача 38 (FFT и обработка сигналов).** Дан сигнал — сумма двух синусоид с шумом (например, суточные и недельные колебания посещаемости):
```python
t = np.linspace(0, 1, 1000, endpoint=False)
signal = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)
noisy = signal + np.random.default_rng(42).normal(0, 0.3, 1000)
```
Выполните: `np.fft.fft(noisy)` — прямое преобразование; `np.fft.fftfreq(len(noisy), d=1/1000)` — частоты; `amplitudes = np.abs(fft) / len(noisy) * 2` — амплитуды; найдите топ-2 частоты через `np.argsort(amplitudes)[-2:]`. Отфильтруйте шум: обнулите все частоты выше 20 Гц и выполните `np.fft.ifft`. Сравните ошибку до и после фильтрации. В комментариях объясните: что такое FFT (как раскладывает сигнал на частоты); почему длины, кратные степени двойки, эффективнее (какой алгоритм используется); как амплитуда связана с коэффициентом синусоиды (почему умножаем на 2 и делим на длину); как FFT применяется к анализу сезонности посещаемости клуба (выявление периодических пиков); что такое спектр и как его интерпретировать.

**Задача 39 (полиномы и интерполяция).** Даны дискретные измерения количества посещений за 5 дней: `x = np.array([0, 1, 2, 3, 4])`, `y = np.array([120, 150, 110, 180, 140])`. Реализуйте: интерполяцию через `np.interp` в точках `[0.5, 1.5, 2.5, 3.5]`; подгонку полинома степени 2 через `np.polynomial.Polynomial.fit(x, y, deg=2)`; предсказание в тех же промежуточных точках; сравните результаты интерполяции и полинома. Дополнительно: `np.interp` с параметрами `left=-1`, `right=-1` для точек за пределами диапазона. В комментариях объясните: чем интерполяция отличается от аппроксимации (интерполяция проходит через все точки, аппроксимация — приближается); почему полином высокой степени может «выстреливать» за пределы данных (феномен Рунге) и как это связано с переобучением; в каком случае `np.interp` (кусочно-линейная) лучше полинома (когда данные шумные или их мало); как интерполяция применяется для восстановления пропущенных дней в отчёте клуба; какие ещё методы интерполяции существуют (сплайны, кубическая интерполяция).

**Задача 40 (`einsum` и структурированные массивы).** Даны: `A` формы `(3, 4)` — матрица «филиалы × типы тренировок»; `B` формы `(4, 5)` — матрица «типы тренировок × дни»; `v` формы `(4,)` — вектор весов типов тренировок. Реализуйте через `np.einsum`: матричное умножение `A @ B`: `np.einsum('ik,kj->ij', A, B)`; умножение матрицы на вектор `A @ v`: `np.einsum('ik,k->i', A, v)`; скалярное произведение `v @ v`: `np.einsum('i,i->', v, v)`; внешнее произведение `v` на себя: `np.einsum('i,j->ij', v, v)`; след квадратной матрицы: `np.einsum('ii->', M)`. Дополнительно создайте структурированный массив посещений:
```python
visits_log = np.array([
    ('2025-01-01', 'Иван Петров', 'йога', 60),
    ('2025-01-02', 'Мария Иванова', 'бокс', 90),
    ('2025-01-03', 'Пётр Сидоров', 'кроссфит', 120),
], dtype=[('date', 'U10'), ('member', 'U30'), ('workout', 'U20'), ('duration', 'i4')])
```
Вычислите среднюю длительность (`visits_log['duration'].mean()`) и отфильтруйте посещения длительнее 60 минут. В комментариях объясните: в каком случае `einsum` читаемее, чем `@` (когда операция нестандартная); почему структурированные массивы удобны для табличных данных, но не заменяют Pandas (какие операции Pandas удобнее); как `einsum` обобщается на тензоры (`'bij,bjk->bik'`) и где это применяется (пакетная обработка); как структурированный массив связан с `@dataclass` из Python OOP (оба хранят разнородные поля, но NumPy — это C-структуры в памяти с фиксированным расположением).